# 03 — Gold layer (Procurement)

Gold tables for the Procurement profile: what we pay suppliers, how exposed we are to any single one, which suppliers are worth keeping.

**Definitions:**
- **Spend** = `l_quantity * ps_supplycost`: what we pay the supplier for the ordered units. The supply cost is taken for the exact (part, supplier) pair of the line item.
- **Inventory value** = `ps_availqty * ps_supplycost`.
- **Supplier with complaints** = `s_comment` contains `Customer ... Complaints`.

| table | grain | used for |
|---|---|---|
| `gold_supplier` | supplier | inventory value, complaints, nation / region |
| `gold_part_supplier` | (part, supplier) offer from partsupp | cost spread, cheapest supplier, single sourcing |
| `gold_supplier_spend_monthly` | month × supplier | spend share, concentration, cost by region over time |

In [0]:
import re

_user = spark.sql("SELECT current_user()").first()[0]
_default_schema = "lakehouse_" + re.sub(r"\W+", "_", _user.split("@")[0]).lower()

dbutils.widgets.text("target_catalog", "workspace", "Target catalog")
dbutils.widgets.text("target_schema", _default_schema, "Target schema")

TARGET_CATALOG = dbutils.widgets.get("target_catalog")
TARGET_SCHEMA = dbutils.widgets.get("target_schema")


def silver(table: str) -> str:
    return f"{TARGET_CATALOG}.{TARGET_SCHEMA}.silver_{table}"


def gold(table: str) -> str:
    return f"{TARGET_CATALOG}.{TARGET_SCHEMA}.gold_{table}"


COMPLAINTS_PATTERN = "%Customer%Complaints%"

print(f"Read from : {TARGET_CATALOG}.{TARGET_SCHEMA}.silver_*")
print(f"Write to  : {TARGET_CATALOG}.{TARGET_SCHEMA}.gold_*")

## Gold tables

### gold_supplier

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold("supplier")} AS
SELECT
    s.s_suppkey AS supplier_id,
    s.s_name AS supplier_name,
    n.n_name AS nation,
    r.r_name AS region,
    s.s_comment AS supplier_comment,
    s.s_comment LIKE '{COMPLAINTS_PATTERN}' AS has_complaints,
    COALESCE(ps.parts_offered, 0) AS parts_offered,
    COALESCE(ps.available_quantity, 0) AS available_quantity,
    COALESCE(ps.inventory_value, 0) AS inventory_value
FROM {silver("supplier")} s
JOIN {silver("nation")} n ON n.n_nationkey = s.s_nationkey
JOIN {silver("region")} r ON r.r_regionkey = n.n_regionkey
LEFT JOIN (
    SELECT
        ps_suppkey,
        COUNT(*) AS parts_offered,
        SUM(ps_availqty) AS available_quantity,
        SUM(ps_availqty * ps_supplycost) AS inventory_value
    FROM {silver("partsupp")}
    GROUP BY ps_suppkey
) ps ON ps.ps_suppkey = s.s_suppkey
""")

display(spark.table(gold("supplier")).limit(10))

### gold_part_supplier

One row per offer in partsupp, including offers that were never ordered.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold("part_supplier")} AS
WITH ordered AS (
    SELECT
        l_partkey,
        l_suppkey,
        COUNT(*) AS order_lines,
        SUM(l_quantity) AS ordered_quantity
    FROM {silver("lineitem")}
    GROUP BY l_partkey, l_suppkey
)
SELECT
    ps.ps_partkey AS part_id,
    p.p_name AS part_name,
    p.p_brand AS brand,
    b.p_mfgr AS manufacturer,
    p.p_type AS part_type,
    p.p_retailprice AS retail_price,
    ps.ps_suppkey AS supplier_id,
    ps.ps_supplycost AS supply_cost,
    ps.ps_availqty AS available_quantity,
    COUNT(*) OVER (PARTITION BY ps.ps_partkey) AS suppliers_offering,
    SUM(CASE WHEN o.order_lines > 0 THEN 1 ELSE 0 END) OVER (PARTITION BY ps.ps_partkey) AS suppliers_ordered,
    MIN(ps.ps_supplycost) OVER (PARTITION BY ps.ps_partkey) AS min_supply_cost,
    MAX(ps.ps_supplycost) OVER (PARTITION BY ps.ps_partkey) AS max_supply_cost,
    ps.ps_supplycost = MIN(ps.ps_supplycost) OVER (PARTITION BY ps.ps_partkey) AS is_cheapest,
    COALESCE(o.order_lines, 0) AS order_lines,
    COALESCE(o.ordered_quantity, 0) AS ordered_quantity,
    COALESCE(o.ordered_quantity, 0) * ps.ps_supplycost AS spend
FROM {silver("partsupp")} ps
JOIN {silver("part")} p ON p.p_partkey = ps.ps_partkey
JOIN {silver("brand")} b ON b.p_brand = p.p_brand
LEFT JOIN ordered o
    ON o.l_partkey = ps.ps_partkey
   AND o.l_suppkey = ps.ps_suppkey
""")

display(spark.table(gold("part_supplier")).limit(10))

### gold_supplier_spend_monthly

Month of the order date × supplier.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold("supplier_spend_monthly")} AS
SELECT
    CAST(DATE_TRUNC('month', o.o_orderdate) AS DATE) AS month,
    l.l_suppkey AS supplier_id,
    r.r_name AS region,
    COUNT(*) AS order_lines,
    SUM(l.l_quantity) AS quantity,
    SUM(l.l_quantity * ps.ps_supplycost) AS spend
FROM {silver("lineitem")} l
JOIN {silver("orders")} o ON o.o_orderkey = l.l_orderkey
JOIN {silver("partsupp")} ps
    ON ps.ps_partkey = l.l_partkey
   AND ps.ps_suppkey = l.l_suppkey
JOIN {silver("supplier")} s ON s.s_suppkey = l.l_suppkey
JOIN {silver("nation")} n ON n.n_nationkey = s.s_nationkey
JOIN {silver("region")} r ON r.r_regionkey = n.n_regionkey
GROUP BY CAST(DATE_TRUNC('month', o.o_orderdate) AS DATE), l.l_suppkey, r.r_name
""")

display(spark.table(gold("supplier_spend_monthly")).limit(10))

## Validation

Gold must reconcile with silver: no rows lost by joins and no double-counting across suppliers of the same part.

In [0]:
TOLERANCE = 0.01


def scalar(query: str):
    return spark.sql(query).first()[0]


silver_spend = scalar(f"""
    SELECT SUM(l.l_quantity * ps.ps_supplycost)
    FROM {silver("lineitem")} l
    JOIN {silver("partsupp")} ps
      ON ps.ps_partkey = l.l_partkey AND ps.ps_suppkey = l.l_suppkey
""")
silver_inventory = scalar(f"SELECT SUM(ps_availqty * ps_supplycost) FROM {silver('partsupp')}")
silver_lines = spark.table(silver("lineitem")).count()
silver_offers = spark.table(silver("partsupp")).count()
silver_suppliers = spark.table(silver("supplier")).count()

checks = [
    ("spend: gold_part_supplier = silver",
     scalar(f"SELECT SUM(spend) FROM {gold('part_supplier')}"), silver_spend),
    ("spend: gold_supplier_spend_monthly = silver",
     scalar(f"SELECT SUM(spend) FROM {gold('supplier_spend_monthly')}"), silver_spend),
    ("inventory value: gold_supplier = silver",
     scalar(f"SELECT SUM(inventory_value) FROM {gold('supplier')}"), silver_inventory),
    ("order lines: gold_part_supplier = silver lineitem",
     scalar(f"SELECT SUM(order_lines) FROM {gold('part_supplier')}"), silver_lines),
    ("order lines: gold_supplier_spend_monthly = silver lineitem",
     scalar(f"SELECT SUM(order_lines) FROM {gold('supplier_spend_monthly')}"), silver_lines),
    ("rows: gold_part_supplier = silver partsupp",
     spark.table(gold("part_supplier")).count(), silver_offers),
    ("rows: gold_supplier = silver supplier",
     spark.table(gold("supplier")).count(), silver_suppliers),
]

results = [
    (name, float(gold_value), float(silver_value), abs(gold_value - silver_value) <= TOLERANCE)
    for name, gold_value, silver_value in checks
]

display(spark.createDataFrame(results, "check_name STRING, gold_value DOUBLE, silver_value DOUBLE, passed BOOLEAN"))

failed = [r[0] for r in results if not r[3]]
assert not failed, f"Gold validation failed: {failed}"
print("OK: all Gold validation checks passed.")